In [1]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente
load_dotenv()

# Importazioni necessarie
from datapizzai.clients import (
    ClientFactory, 
    OpenAIClient,
    AnthropicClient, 
    GoogleClient,
    MistralClient,
    AzureOpenAIClient
)
from datapizzai.clients.factory import Provider

## Custom Client

In [7]:
import os
import requests
from typing import Optional, Union, List
from pydantic import BaseModel

from datapizzai.type import TextBlock
from datapizzai.memory import Memory


class SimpleResponse(BaseModel):
    text: str
    prompt_tokens_used: int = 0
    completion_tokens_used: int = 0
    stop_reason: str = "stop"


class OllamaGemmaClient:
    """Adapter minimale per Ollama Chat API con modello Gemma."""

    def __init__(
        self,
        model: str = "gemma3n:e2b",
        system_prompt: Optional[str] = None,
        temperature: float = 0.7,
        base_url: str = "http://localhost:11434",
    ):
        self.model = model
        self.system_prompt = system_prompt or ""
        self.temperature = temperature
        self.base_url = base_url.rstrip("/")

    def _build_messages(
        self,
        input: Optional[Union[str, List[TextBlock]]] = None,
        memory: Optional[Memory] = None,
    ) -> List[dict]:
        messages: List[dict] = []
        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})
        if memory is not None:
            for turn in memory.memory:
                role = turn.role.value if hasattr(turn.role, "value") else str(turn.role)
                content = " ".join(getattr(b, "content", "") for b in turn.blocks)
                if content:
                    messages.append({"role": role, "content": content})
        if isinstance(input, str) and input:
            messages.append({"role": "user", "content": input})
        elif isinstance(input, list) and input:
            user_text = " ".join(b.content for b in input if isinstance(b, TextBlock))
            if user_text:
                messages.append({"role": "user", "content": user_text})
        return messages

    def invoke(
        self,
        input: Optional[Union[str, List[TextBlock]]] = None,
        memory: Optional[Memory] = None,
    ) -> SimpleResponse:
        messages = self._build_messages(input=input, memory=memory)
        payload = {
            "model": self.model,
            "messages": messages,
            "stream": False,
            "options": {"temperature": self.temperature},
        }
        try:
            r = requests.post(f"{self.base_url}/api/chat", json=payload, timeout=120)
            r.raise_for_status()
            data = r.json()
            # Risposta attesa: {"message": {"role": "assistant", "content": "..."}, ...}
            text = data.get("message", {}).get("content") or str(data)
        except Exception as e:
            text = f"Errore Ollama: {e}"
        return SimpleResponse(text=text)


# Test rapido del setup
if __name__ == "__main__":
    client = OllamaGemmaClient(
        model="gemma3n:e2b",  # Sostituisci con il tag del tuo modello Gemma locale
        system_prompt="Sei un assistente AI utile e conciso.",
        temperature=0.7,
    )
    resp = client.invoke("Ciao! Presentati brevemente in due frasi.")
    print(f"Risposta: {resp.text}")

Risposta: Ciao! Sono un modello linguistico di grandi dimensioni, addestrato da Google. Sono qui per aiutarti con informazioni, idee e compiti testuali.
